# 43. The ratio block on the two encoded frames it has not met

**One variable in both arms: the ratio block, on or off.**

| arm | baseline | that row's CV |
|---|---|---|
| `lgb_te_fe` | row 17 `lgbm_bag08_seed42_te` | 0.966782 |
| `cat_te_fe` | row 87 `cat_te_n4000` | 0.967166 |

Row 93 put the block on XGBoost's encoded frame and it was worth **+0.000906**, the largest of
the four arms in notebook `40` and the one predicted to be null. These are the two encoded frames
it has never been run against, and together with row 93 they complete the grid: block on and off,
against all three learners, on both the raw and the encoded representation.

## Why this is worth a kernel and what it is probably worth

The block is the only feature idea in this repo that has ever paid on the encoded frame. But row
94 measured what happens when several ratio-block members go into the stack together: they are
**subadditive**, at 0.39x the sum of their individual contributions, because they carry the same
new information and compete for the same weight.

So the honest expectation is two good single models and a **small** stack gain, not a repeat of
row 94's +0.000604. This is being run because the two models are cheap and are likely to be strong
members, not because the stack is expected to move much.

## The CatBoost arm uses the fitted budget, deliberately

`cat_te_fe` runs at **4,000 iterations** against row 87, not at the 2,000 of row 26. Rows 85 to 89
established 4,000 as the bracketed optimum on this frame, so row 87 is the correct baseline and
holding the budget there is what keeps this one variable. `leaf_estimation_iterations` and
`max_ctr_complexity` are pinned at 10 and 4 for the reason recorded on 2026-08-21, and
`thread_count` is 6 to match row 87.

## The prediction, written before the run

**Both arms positive, between +0.0005 and +0.0012**, by analogy with row 93's +0.000906 on the
third encoded frame.

I am stating a magnitude reluctantly. **Three consecutive magnitude predictions have been wrong**,
in rows 39, 40 and 42, the last by a factor of twenty to sixty. The difference here is that this
is not a mechanism argument invented for the occasion: it is the same block, on the same kind of
frame, one learner over. That is interpolation between measured points rather than extrapolation
from a story, which is the only kind of prediction this repo has a decent record on.

**The honest case against.** LightGBM and CatBoost both route NaN natively and both handle the
encoded columns well, and the ratio columns are 18 to 39 percent missing. If row 93's gain came
partly from XGBoost's particular NaN handling rather than from the block, these arms will be
smaller.

## What this decides

Nothing about the stack. Membership is a separate notebook and a separate ledger row. No
submission csv.

In [ ]:
# One flag. The run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# The budget convention every baseline used: learning_rate * n_estimators = 100.
LR = 0.05
N_EST = 2000
MAX_DEPTH = 6
BENCH_EST = 200
PROBE_FOLD = 0
THREADS = -1

# CatBoost auto-selects these from `iterations` (1 and 1 up to 150, 10 and 4 from 200).
# Pinned so the cat arm is the same model family as row 78 regardless of budget. See
# NOTES.md, 2026-08-21.
LEAF_EST_ITERS = 10
MAX_CTR_COMPLEXITY = 4

ARMS = ["lgb_te_fe", "cat_te_fe"]
# Row 87 is CatBoost at its fitted budget, so the cat arm must run there too.
CAT_ITERS = 4000
CAT_THREADS = 6

# The row each arm changes exactly one variable against: the ratio block, on or off.
BASELINES = {
    "lgb_te_fe": ("te_bag42", 0.966782, "row 17 lgbm_bag08_seed42_te"),
    "cat_te_fe": ("cat_te_n4000", 0.967166, "row 87 cat_te_n4000"),
}
ENCODED_ARMS = {"lgb_te_fe", "cat_te_fe"}

MAX_HOURS = 9.0
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

print(f"SMOKE = {SMOKE}   lr {LR}   n_estimators {N_EST}")
print(f"arms: {ARMS}")

## Stage 1. Data, folds, leak checklist

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

## Stage 2. The ratio block

Thirteen columns, every one a pure function of other feature columns. Nothing here touches the
target, which is the whole leak argument for the three view arms, and it is asserted rather than
described.

`safe_div` sends a zero denominator to NaN rather than to infinity, and any infinity that
survives is mapped to NaN afterwards. All three learners route NaN natively, so a missing
component produces a missing ratio rather than a fabricated number, which is the behaviour the
raw columns already have.

In [ ]:
DST, SM, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                   "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    """The 13 composition features. A pure function of the feature columns."""
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


def frame(df, fe, for_cb=False):
    """Row 68/69/70's frame, optionally with the ratio block appended."""
    X = df[COLS].copy()
    for c in CAT:
        if for_cb:
            X[c] = X[c].astype(object).fillna("__missing__").astype(str)
        else:
            X[c] = X[c].astype("category")
    if fe:
        X = pd.concat([X, ratio_block(df)], axis=1)
    return X


# The block must be bit-identical under a permutation of y. DataFrame.equals treats NaN in
# the same position as equal, which plain `==` does not: on pandas 3.0 the new string dtype
# preserves NA, so an `==` comparison silently reports every missing cell as unequal. Same
# version trap that eats target encoders, met here in a check.
rng = np.random.default_rng(0)
train_perm = train.copy()
train_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
pure = ratio_block(train_perm).equals(ratio_block(train))
print(f"ratio block is a pure function of the features, not of y: {pure}")

rb = ratio_block(train)
print(f"{len(RATIO_COLS)} ratio columns, "
      f"{len(COLS)} raw -> {len(COLS) + len(RATIO_COLS)} with the block on")
print(f"columns match the declared list: {list(rb.columns) == RATIO_COLS}")
print()
print(f"{'feature':<20}{'missing':>10}{'mean':>12}{'sd':>12}")
for c in RATIO_COLS:
    print(f"  {c:<18}{rb[c].isna().mean():>9.2%}{rb[c].mean():>12.4f}{rb[c].std():>12.4f}")

# The generator fingerprint, restated as a measurement rather than a citation.
sl = rb["slack"].dropna()
print()
print(f"slack = daily - (social + gaming + work_study), on {len(sl):,} complete rows:")
print(f"  negative (constraint violated): {(sl < -1e-9).mean():.4%}")
print(f"  median {sl.median():+.3f} h")
print("  a hard floor at zero is the generator's doing; the source file does not obey it.")
BLOCK_OK = bool(pure and list(rb.columns) == RATIO_COLS)
del rb, train_perm
gc.collect()

## Stage 3. The encoder, for the `xgb_te_fe` arm only

Copied verbatim from `34_xgb_depth.ipynb` and fingerprinted against `13_target_encoding.ipynb`,
because a silently different encoder would make the encoded arm two variables rather than one
and would not show up anywhere in the score.

In [ ]:
X = train[COLS].copy()
X_test = test[COLS].copy()

def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

Run verbatim from `34`, on the same encoder, so the numbers are directly comparable to the ones
`13` recorded. The encoder is the only thing in this notebook capable of leaking; the ratio
block cannot, and that was asserted in stage 2.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 4. Bench, determinism, and the projection

In [ ]:
def make_lgb(n_est):
    # Row 9 and row 17's configuration.
    return lgb.LGBMClassifier(
        objective="binary", metric="auc", learning_rate=LR, n_estimators=n_est,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=SEED, n_jobs=THREADS, verbose=-1,
        deterministic=True, force_row_wise=True,
    )


def make_xgb(n_est):
    # Row 38's configuration.
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=THREADS, verbosity=0,
    )


def make_cat(n_est):
    # Row 87's configuration: the fitted 4,000-iteration budget, both auto-tuned parameters
    # pinned, and thread_count 6 to match the row this arm is compared against. Under SMOKE
    # the caller's small n_est and single thread win, since no ledger number comes from it.
    return cb.CatBoostClassifier(
        iterations=n_est if SMOKE else CAT_ITERS,
        learning_rate=LR, random_seed=SEED,
        leaf_estimation_iterations=LEAF_EST_ITERS,
        max_ctr_complexity=MAX_CTR_COMPLEXITY,
        thread_count=THREADS if SMOKE else CAT_THREADS,
        allow_writing_files=False, verbose=0,
    )


def build_arm(arm, fold, want_test=False):
    """Feature frames for one arm and one fold. The ratio block is ON in every arm here;
    the baseline each is compared against is the ledger row with it off."""
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    if arm in ENCODED_ARMS:
        # Encode inside the fold, then append the block. build() takes the raw frame.
        Xtr, Xva, Xte = build(X, y, tr, va, X_test if want_test else None)
        if arm.startswith("cat"):
            # CatBoost takes categoricals as strings with no NaN, exactly as row 87 did.
            for c in CAT:
                Xtr[c] = Xtr[c].astype(object).fillna("__NA__").astype(str)
                Xva[c] = Xva[c].astype(object).fillna("__NA__").astype(str)
        else:
            for c in CAT:
                Xtr[c] = Xtr[c].astype("category")
                Xva[c] = Xva[c].astype("category")
        rb_tr = ratio_block(train.iloc[tr]).reset_index(drop=True)
        rb_va = ratio_block(train.iloc[va]).reset_index(drop=True)
        Xtr = pd.concat([Xtr, rb_tr], axis=1)
        Xva = pd.concat([Xva, rb_va], axis=1)
        if want_test:
            if arm.startswith("cat"):
                for c in CAT:
                    Xte[c] = Xte[c].astype(object).fillna("__NA__").astype(str)
            else:
                for c in CAT:
                    Xte[c] = Xte[c].astype("category")
            Xte = pd.concat([Xte, ratio_block(test).reset_index(drop=True)], axis=1)
        idx = [Xtr.columns.get_loc(c) for c in CAT] if arm.startswith("cat") else None
        return tr, va, Xtr, Xva, Xte, idx
    raise AssertionError("every arm in this notebook is on the encoded frame")


def fit_arm(arm, Xtr, ytr, idx, n_est):
    if arm.startswith("cat"):
        m = make_cat(n_est)
        m.fit(Xtr, ytr, cat_features=idx)
    elif arm.startswith("lgb"):
        m = make_lgb(n_est)
        m.fit(Xtr, ytr)
    else:
        m = make_xgb(n_est)
        m.fit(Xtr, ytr)
    return m


def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


bench = {}
for arm in ARMS:
    tr, va, Xtr, Xva, _, idx = build_arm(arm, PROBE_FOLD)
    t0 = time.time()
    m = fit_arm(arm, Xtr, y[tr], idx, BENCH_EST)
    secs = time.time() - t0
    p1 = m.predict_proba(Xva)[:, 1]
    m2 = fit_arm(arm, Xtr, y[tr], idx, BENCH_EST)
    drift = float(np.max(np.abs(p1 - m2.predict_proba(Xva)[:, 1])))
    bench[arm] = secs
    print(f"{arm:12} bench {BENCH_EST}: AUC {roc_auc_score(y[va], p1):.6f} in {hhmm(secs)}, "
          f"{Xtr.shape[1]} features, repeat drift {drift:.3e} "
          f"{'OK' if drift == 0.0 else 'NOT DETERMINISTIC'}")
    del m, m2, Xtr, Xva
    gc.collect()

projected = sum(bench[a] / BENCH_EST * N_EST * 5 for a in ARMS)
print(f"\nprojected full run: {hhmm(projected)} for {len(ARMS)} arms x 5 folds")
GO = projected < MAX_HOURS * 3600 or SMOKE
print("within budget" if GO else f"OVER the {MAX_HOURS}h guard, not starting.")

## Stage 5. The run

In [ ]:
assert LEAK_OK and CLEAN, "leak checks failed"
assert ENCODER_MATCH, "encoder does not match 13, the encoded arm would not be one variable"
assert BLOCK_OK, "the ratio block is not a pure function of the features"
assert GO, "over the time guard"
if not SMOKE:
    assert ALIGNED, "fold sha mismatch"

pre = "SMOKE_" if SMOKE else ""
results = {}
t_start = time.time()
for arm in ARMS:
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    per = []
    t_arm = time.time()
    for f in range(5):
        tr, va, Xtr, Xva, Xte, idx = build_arm(arm, f, want_test=True)
        m = fit_arm(arm, Xtr, y[tr], idx, N_EST)
        oof[va] = m.predict_proba(Xva)[:, 1]
        tst[f] = m.predict_proba(Xte)[:, 1]
        per.append(float(roc_auc_score(y[va], oof[va])))
        del m, Xtr, Xva, Xte
        gc.collect()
    results[arm] = {"oof": oof, "test": tst.mean(axis=0), "per": np.array(per)}
    print(f"{arm:12} CV {np.mean(per):.6f} +/- {np.std(per):.6f}  [{hhmm(time.time() - t_arm)}]")
    np.save(OUT / f"{pre}{arm}_oof.npy", oof)
    np.save(OUT / f"{pre}{arm}_test.npy", tst.mean(axis=0))

print(f"\nall arms done in {hhmm(time.time() - t_start)}")

In [ ]:
def load_base(stem):
    """The baseline's saved OOF vector, for a PAIRED comparison rather than a comparison
    of means. It lives in the attached dataset on Kaggle and in artifacts/oof locally, so
    it is found by name through locate(). Looking only in OUT was a bug: on Kaggle OUT is
    /kaggle/working, which is empty at read time, and the paired statistics would have
    silently degraded to a difference of two means with no fold count and no sd."""
    for name in (f"{stem}_oof.npy", f"{stem}.npy"):
        try:
            return np.load(locate(name))
        except FileNotFoundError:
            continue
    return None


print("Paired against each arm's own baseline. One variable each: the ratio block.\n")
print(f"{'arm':12} {'CV':>10} {'baseline':>10} {'paired':>11} {'sd':>10} {'folds':>7} {'t(4)':>8}")
summary = {}
for arm in ARMS:
    stem, base_cv, base_row = BASELINES[arm]
    per = results[arm]["per"]
    bv = load_base(stem)
    if bv is not None:
        # ROW_IDX maps this run's rows back into the full-length saved vectors, so the
        # paired path is exercised under SMOKE too. The smoke numbers are meaningless;
        # running the code is the point, because a bug here would otherwise stay hidden
        # until the full run and cost the whole kernel.
        bper = np.array([roc_auc_score(y[folds == f], bv[ROW_IDX][folds == f])
                         for f in range(5)])
        note = f"baseline refit check {bper.mean() - base_cv:+.2e}"
    else:
        bper = None
        note = "baseline vector NOT FOUND, falling back to the ledger mean"
    if bper is not None:
        d = per - bper
        sd = d.std(ddof=1)
        t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("nan")
        print(f"{arm:12} {per.mean():10.6f} {bper.mean():10.6f} {d.mean():+11.6f} "
              f"{sd:10.6f} {int((d > 0).sum()):5d}/5 {t:8.2f}")
        summary[arm] = (per.mean(), d.mean(), int((d > 0).sum()))
    else:
        print(f"{arm:12} {per.mean():10.6f} {base_cv:10.6f} {per.mean() - base_cv:+11.6f} "
              f"{'':>10} {'':>7} {'':>8}   ({note})")
        summary[arm] = (per.mean(), per.mean() - base_cv, -1)

print()
if SMOKE:
    print("SMOKE: the paired columns above are MEANINGLESS, and not merely noisy. The arms")
    print("  trained on 20,000 rows at 200 iterations; the baseline vectors they are paired")
    print("  against were produced at full size and full budget. The comparison is invalid by")
    print("  construction. It is printed so the code path runs, not so the numbers are read.")
else:
    print("Both arms are on the encoded frame, so there is no 2 by 2 here. Together with")
    print("row 93 they complete the grid: the block against all three learners on the")
    print("encoded representation, and against all three on the raw one in rows 90 to 92.")
    print()
    print(f"  {'arm':12} {'paired':>11}")
    for a in ARMS:
        print(f"  {a:12} {summary[a][1]:+11.6f}   this run")
    print(f"  {'xgb_te_fe':12} {0.000906:+11.6f}   row 93, encoded")
    print(f"  {'cat_raw_fe':12} {0.000761:+11.6f}   row 92, raw")
    print(f"  {'xgb_raw_fe':12} {0.000448:+11.6f}   row 91, raw")
    print(f"  {'lgb_raw_fe':12} {0.000356:+11.6f}   row 90, raw")

In [ ]:
print("ledger lines:")
for arm in ARMS:
    per = results[arm]["per"]
    print(f"  name    {arm}\n  cv_mean {per.mean():.6f}\n  cv_std  {per.std():.6f}")
print(f"\n  leak checks {'PASS' if (LEAK_OK and CLEAN) else 'FAILED'}, "
      f"encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"ratio block pure {BLOCK_OK}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is decided in a separate notebook, which changes")
print("a different variable and gets its own ledger row.")